<a href="https://colab.research.google.com/github/Karol315/Hackathon_Task_1/blob/main/nn_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
from skfp.fingerprints import MACCSFingerprint, ECFPFingerprint
from joblib import Parallel, delayed

In [23]:
#!pip install scikit-fingerprints

In [24]:

# ==========================================
# FUNKCJA: Parsowanie hierarchii z pliku .obo
# ==========================================
def parse_obo_hierarchy(file_path):
    """
    Czyta plik .obo i zwraca słownik relacji: { 'Rodzic': ['Dziecko1', 'Dziecko2', ...] }
    """
    parent_child_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        current_term = None
        for line in f:
            line = line.strip()
            if line.startswith('[Term]'):
                current_term = None
            elif line.startswith('id:'):
                current_term = line.split('id:')[1].strip()
                if current_term not in parent_child_map:
                    parent_child_map[current_term] = []
            elif line.startswith('is_a:'):
                parent = line.split('is_a:')[1].split('!')[0].strip()
                if parent not in parent_child_map:
                    parent_child_map[parent] = []
                if current_term:
                    parent_child_map[parent].append(current_term)
    return parent_child_map



In [25]:
# ==========================================
# KROK 1: Wczytanie i podział danych
# ==========================================
print("1. Wczytywanie danych...")
train_full_df = pd.read_parquet('/content/Hackathon_Task_1/Hackathon_task_1/data/chebi_dataset_train.parquet')

# Wyodrębnienie kolumn
smiles_full = train_full_df['SMILES'].tolist()
y_full = train_full_df.drop(columns=['SMILES', 'mol_id']).values
class_names = train_full_df.drop(columns=['SMILES', 'mol_id']).columns.tolist()

print("   -> Dzielenie na zbiór treningowy i testowy (80/20)...")
smiles_train, smiles_test, y_train, y_test = train_test_split(
    smiles_full, y_full, test_size=0.1, random_state=42
)

1. Wczytywanie danych...
   -> Dzielenie na zbiór treningowy i testowy (80/20)...


In [27]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
from skfp.fingerprints import MACCSFingerprint, ECFPFingerprint
from joblib import Parallel, delayed

# Inicjalizacja z n_jobs=-1 dla pełnej mocy procesora
maccs_gen = MACCSFingerprint(n_jobs=-1)
# Zostawiam Morgana (ECFP), bo genialnie łapie strukturę
morgan_gen = ECFPFingerprint(radius=2, fp_size=2048, n_jobs=-1)

# Twój zoptymalizowany zestaw deskryptorów
def get_advanced_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(10)

    try:
        # 7 PhysChem Descriptors
        wt = Descriptors.MolWt(mol) / 1000            # Normalized Weight
        logp = Descriptors.MolLogP(mol) / 10          # Hydrophobicity
        hbd = Lipinski.NumHDonors(mol) / 10           # H-Bond Donors
        hba = Lipinski.NumHAcceptors(mol) / 10        # H-Bond Acceptors
        tpsa = Descriptors.TPSA(mol) / 200            # Polar Surface Area
        fsp3 = Lipinski.FractionCSP3(mol)             # Complexity (sp3 fraction)

        # 4 Topological/Ring Descriptors
        rings = Lipinski.RingCount(mol) / 10          # Total Rings
        arom_rings = Lipinski.NumAromaticRings(mol) / 10
        hetero_atoms = Lipinski.NumHeteroatoms(mol) / 20
        bridgeheads = rdMolDescriptors.CalcNumBridgeheadAtoms(mol) / 5

        return [wt, logp, hbd, hba, tpsa, fsp3, rings, arom_rings, hetero_atoms, bridgeheads]
    except:
        return np.zeros(10)

def extract_all_features(smiles_list):
    print(f"Przetwarzanie {len(smiles_list)} cząsteczek...")

    print("Obliczanie kluczy MACCS...")
    X_maccs = maccs_gen.transform(smiles_list)

    print("Obliczanie fingerprintów Morgana (ECFP)...")
    X_morgan = morgan_gen.transform(smiles_list)

    print("Obliczanie Twoich zaawansowanych deskryptorów na wszystkich rdzeniach...")
    X_desc_list = Parallel(n_jobs=-1)(delayed(get_advanced_descriptors)(s) for s in smiles_list)
    X_desc = np.array(X_desc_list)

    print("Łączenie wszystkich cech w jedną macierz...")
    X = np.concatenate([X_maccs, X_morgan, X_desc], axis=1)

    # Ostatnie zabezpieczenie przed błędem z PyTorcha
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Gotowe! Ostateczny kształt macierzy: {X.shape}")
    return X

# --- URUCHOMIENIE ---
X_train = extract_all_features(smiles_train)
X_test = extract_all_features(smiles_test)

Przetwarzanie 30301 cząsteczek...
Obliczanie kluczy MACCS...
Obliczanie fingerprintów Morgana (ECFP)...
Obliczanie Twoich zaawansowanych deskryptorów na wszystkich rdzeniach...
Łączenie wszystkich cech w jedną macierz...
Gotowe! Ostateczny kształt macierzy: (30301, 2224)
Przetwarzanie 3367 cząsteczek...
Obliczanie kluczy MACCS...
Obliczanie fingerprintów Morgana (ECFP)...
Obliczanie Twoich zaawansowanych deskryptorów na wszystkich rdzeniach...
Łączenie wszystkich cech w jedną macierz...
Gotowe! Ostateczny kształt macierzy: (3367, 2224)


In [32]:
import os
import random
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# ==========================================
# FUNKCJE POMOCNICZE
# ==========================================

def set_seed(seed=42):
    """Wymusza pełną reprodukowalność wyników."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Dla wielu GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def init_weights(m):
    """Inicjalizacja wag Kaiming (He) zoptymalizowana dla ReLU."""
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

# ==========================================
# KROK 3: Trening modelu (Sieć Neuronowa / PyTorch GPU)
# ==========================================
print("3. Trenowanie modelu (PyTorch GPU)...")

# Ustawienie ziarna przed rozpoczęciem jakichkolwiek operacji losowych
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   -> Używane urządzenie: {device}")

valid_cols_mask = np.array([len(np.unique(y_train[:, i])) > 1 for i in range(y_train.shape[1])])
constant_classes = {class_names[i]: y_train[0, i] for i in range(len(class_names)) if not valid_cols_mask[i]}
print(f"   -> Pomijanie {len(constant_classes)} stałych klas w zbiorze treningowym...")

y_train_filtered = y_train[:, valid_cols_mask]
y_test_filtered = y_test[:, valid_cols_mask]
valid_class_names = [class_names[i] for i in range(len(class_names)) if valid_cols_mask[i]]

# Konwersja danych wejściowych na Tensory
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_filtered, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test_filtered, dtype=torch.float32)

# Przygotowanie paczek danych (batch_size = 64)
batch_size = 64
# UWAGA: generator jest ustawiany, aby DataLoader również był w pełni reprodukowalny przy tasowaniu
g = torch.Generator()
g.manual_seed(42)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True, generator=g)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

# Definicja sieci
model = nn.Sequential(
    nn.Linear(X_train_t.shape[1], 1024),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, y_train_t.shape[1])
).to(device)

# Aplikacja inicjalizacji wag do modelu
model.apply(init_weights)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Scheduler zmniejszający LR
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# Pętla treningowa
epochs = 200

best_model_wts = copy.deepcopy(model.state_dict())
best_test_f1 = 0.0

for epoch in range(epochs):
    # --- FAZA TRENINGOWA ---
    model.train()
    train_loss = 0.0
    train_preds, train_targets = [], []

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        train_preds.append(preds.cpu().numpy())
        train_targets.append(targets.cpu().numpy())

    epoch_train_loss = train_loss / len(train_loader.dataset)
    epoch_train_f1 = f1_score(np.vstack(train_targets), np.vstack(train_preds), average='macro', zero_division=0)

    # --- FAZA WALIDACYJNA (TESTOWA) ---
    model.eval()
    test_loss = 0.0
    test_preds, test_targets = [], []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item() * inputs.size(0)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            test_preds.append(preds.cpu().numpy())
            test_targets.append(targets.cpu().numpy())

    epoch_test_loss = test_loss / len(test_loader.dataset)
    epoch_test_f1 = f1_score(np.vstack(test_targets), np.vstack(test_preds), average='macro', zero_division=0)

    # --- LOGOWANIE I AKTUALIZACJE ---
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoka [{epoch+1:3d}/{epochs}] | LR: {current_lr:.6f} | "
          f"Train Loss: {epoch_train_loss:.4f}, Train F1: {epoch_train_f1:.4f} | "
          f"Test Loss: {epoch_test_loss:.4f}, Test F1: {epoch_test_f1:.4f}")

    scheduler.step(epoch_test_f1)

    if epoch_test_f1 > best_test_f1:
        best_test_f1 = epoch_test_f1
        best_model_wts = copy.deepcopy(model.state_dict())

print(f"\nTrening zakończony. Ładowanie wag dla Test F1: {best_test_f1:.4f}...")
model.load_state_dict(best_model_wts)

3. Trenowanie modelu (PyTorch GPU)...
   -> Używane urządzenie: cuda
   -> Pomijanie 1 stałych klas w zbiorze treningowym...
Epoka [  1/200] | LR: 0.000500 | Train Loss: 0.0873, Train F1: 0.2218 | Test Loss: 0.0410, Test F1: 0.4013
Epoka [  2/200] | LR: 0.000500 | Train Loss: 0.0399, Train F1: 0.4930 | Test Loss: 0.0320, Test F1: 0.5863
Epoka [  3/200] | LR: 0.000500 | Train Loss: 0.0316, Train F1: 0.6114 | Test Loss: 0.0292, Test F1: 0.6727
Epoka [  4/200] | LR: 0.000500 | Train Loss: 0.0267, Train F1: 0.6773 | Test Loss: 0.0269, Test F1: 0.7030
Epoka [  5/200] | LR: 0.000500 | Train Loss: 0.0231, Train F1: 0.7225 | Test Loss: 0.0253, Test F1: 0.7337
Epoka [  6/200] | LR: 0.000500 | Train Loss: 0.0205, Train F1: 0.7551 | Test Loss: 0.0246, Test F1: 0.7576
Epoka [  7/200] | LR: 0.000500 | Train Loss: 0.0184, Train F1: 0.7800 | Test Loss: 0.0238, Test F1: 0.7760
Epoka [  8/200] | LR: 0.000500 | Train Loss: 0.0167, Train F1: 0.8022 | Test Loss: 0.0242, Test F1: 0.7789
Epoka [  9/200] | L

<All keys matched successfully>

In [33]:

# ==========================================
# KROK 4: Post-processing (Logika DAG)
# ==========================================
print("5. Wgrywanie hierarchii i naprawa niekonsekwencji...")
parent_child_map = parse_obo_hierarchy('/content/Hackathon_Task_1/Hackathon_task_1/data/chebi_classes.obo')

corrections_made = 0
for parent, children in parent_child_map.items():
    if parent in class_names:
        for child in children:
            if child in class_names:
                mask = preds_df[child] > preds_df[parent]
                if mask.any():
                    preds_df.loc[mask, child] = preds_df.loc[mask, parent]
                    corrections_made += mask.sum()

print(f"   -> Wprowadzono {corrections_made} poprawek logicznych.")

5. Wgrywanie hierarchii i naprawa niekonsekwencji...
   -> Wprowadzono 87583 poprawek logicznych.


In [34]:
# ==========================================
# KROK 5: Binarizacja i ewaluacja (F1 Macro)
# ==========================================
print("6. Binarne formatowanie i obliczanie F1 Macro...")
final_preds_binary = (preds_df >= 0.5).astype(int)

# Konwersja do numpy array w celu obliczenia błędu
y_pred_array = final_preds_binary.values

# Obliczenie Macro-averaged F1 score
f1_macro = f1_score(y_test, y_pred_array, average='macro')

print("="*40)
print(f" WYNIK WALIDACJI (Macro F1 Score): {f1_macro:.4f}")
print("="*40)

6. Binarne formatowanie i obliczanie F1 Macro...
 WYNIK WALIDACJI (Macro F1 Score): 0.8104


In [13]:
####################################################################################################################

In [46]:
train_df = pd.read_parquet('/content/Hackathon_Task_1/Hackathon_task_1/data/chebi_dataset_train.parquet')

# Wyodrębnienie kolumn
smiles_train = train_df['SMILES'].tolist()
y_train = train_df.drop(columns=['SMILES', 'mol_id']).values
class_names = train_df.drop(columns=['SMILES', 'mol_id']).columns.tolist()

test_df = pd.read_parquet('/content/Hackathon_Task_1/Hackathon_task_1/data/chebi_dataset_test_empty.parquet')
smiles_test = test_df['SMILES'].tolist()

In [47]:
test_df

,mol_id,SMILES
0,mol_6861,OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...
1,mol_29793,O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...
2,mol_26953,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...
3,mol_26053,C=C1CCC(C(C)C)CC1
4,mol_18653,O=C(O)C1CCCCC1=O
...,...,...
11218,mol_13480,Oc1cc2cc[nH]c2cc1O
11219,mol_42067,CNCC(=O)OCc1cccnc1N(C)C(=O)OC(C)[n+]1cnn(C[C@]...
11220,mol_43978,N[C@@H](CCC(=O)[O-])C(O)=Nc1ccc2ccccc2c1
11221,mol_16268,COc1cc(-c2[o+]c3cc(O)cc(O)c3cc2O[C@@H]2O[C@H](...


In [59]:
# X_train = extract_all_features(smiles_train)
X_test = extract_all_features(smiles_test)
len(X_test)

Przetwarzanie 11223 cząsteczek...
Obliczanie kluczy MACCS...
Obliczanie fingerprintów Morgana (ECFP)...
Obliczanie Twoich zaawansowanych deskryptorów na wszystkich rdzeniach...
Łączenie wszystkich cech w jedną macierz...
Gotowe! Ostateczny kształt macierzy: (11223, 2224)


11223

In [62]:
# ==========================================
# KROK 4: Generowanie predykcji (na najlepszych wagach)
# ==========================================

# 1. Konwersja nowego zbioru testowego na Tensor
X_test_t = torch.tensor(X_test, dtype=torch.float32)

# 2. Utworzenie loadera (TensorDataset zawiera teraz tylko jeden element)
# Używamy tego samego batch_size co w treningu
pred_loader = DataLoader(TensorDataset(X_test_t), batch_size=batch_size, shuffle=False)

# Ustawienie modelu w tryb ewaluacji (wyłącza Dropout!)
model.eval()
preds_list = []

# Wyłączamy obliczanie gradientów, żeby oszczędzić pamięć i przyspieszyć proces
with torch.no_grad():
    for inputs in pred_loader:
        # DataLoader zwraca krotkę, więc pobieramy pierwszy (i jedyny) element [0]
        inputs = inputs[0].to(device)

        # Predykcja modelu (tzw. logity)
        outputs = model(inputs)

        # Konwersja na prawdopodobieństwa (0.0 - 1.0) za pomocą Sigmoid
        probs = torch.sigmoid(outputs)
        preds_list.append(probs.cpu().numpy())

# 3. Połączenie wszystkich paczek w jedną macierz predykcji
proba_preds_filtered = np.vstack(preds_list)

# 4. Odtworzenie pełnej ramki danych z wynikami
preds_df = pd.DataFrame(proba_preds_filtered, columns=valid_class_names)

# Przywrócenie klas, które miały stałą wartość (były ignorowane w treningu)
for cls_name, const_val in constant_classes.items():
    preds_df[cls_name] = float(const_val)

# Uporządkowanie kolumn, aby były w tej samej kolejności co oryginalnie
preds_df = preds_df[class_names]

print("Gotowe! Predykcje znajdują się w zmiennej `preds_df`.")

4. Generowanie predykcji na nowym zbiorze X_test...
Gotowe! Predykcje znajdują się w zmiennej `preds_df`.


In [63]:

# ==========================================
# KROK 5: Post-processing (Logika DAG)
# ==========================================
print("5. Wgrywanie hierarchii i naprawa niekonsekwencji...")
parent_child_map = parse_obo_hierarchy('/content/Hackathon_Task_1/Hackathon_task_1/data/chebi_classes.obo')

corrections_made = 0
for parent, children in parent_child_map.items():
    if parent in class_names:
        for child in children:
            if child in class_names:
                mask = preds_df[child] > preds_df[parent]
                if mask.any():
                    preds_df.loc[mask, child] = preds_df.loc[mask, parent]
                    corrections_made += mask.sum()

print(f"   -> Wprowadzono {corrections_made} poprawek logicznych.")

5. Wgrywanie hierarchii i naprawa niekonsekwencji...
   -> Wprowadzono 1950575 poprawek logicznych.


In [64]:
final_preds_binary = (preds_df >= 0.5).astype(int)

final_submission = pd.concat([
    test_df.reset_index(drop=True),
    final_preds_binary.reset_index(drop=True)
], axis=1)


final_submission.head()


,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_6861,OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...,1,1,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_29793,O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...,1,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,mol_26953,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_26053,C=C1CCC(C(C)C)CC1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_18653,O=C(O)C1CCCCC1=O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [65]:

# 4. (Opcjonalnie) Zapis od razu do formatu Parquet
final_submission.to_parquet('nn_test_result_submission_v2.parquet', index=False)